# 2.1 Workflow - Handoffs 交接

- **对应文档**: [workflow_handoffs](https://doc.agentscope.io/tutorial/workflow_handoffs.html)
- **本讲目标**: 理解多 Agent 之间通过 Handoff 进行控制权交接的流程与用法。
- **前置知识**: 02_react_agent, 03_message。

## 学习要点

- Handoff 用于在对话中将「谁在回复」从一个 Agent 切换到另一个。
- 参考官方 doc 中的 handoff 示例，实现简单的双 Agent 交接流程。

In [11]:
# 在此按 doc 编写 handoff 示例：创建两个 Agent，通过 handoff 传递对话权
import asyncio
import os

from agentscope.agent import ReActAgent
from agentscope.formatter import DashScopeChatFormatter
from agentscope.memory import InMemoryMemory
from agentscope.message import Msg
from agentscope.model import DashScopeChatModel
from agentscope.tool import (
    ToolResponse,
    Toolkit,
    execute_python_code,
)
# ... 参考 https://doc.agentscope.io/tutorial/workflow_handoffs.html
print("请参考 workflow_handoffs 文档完成 Handoffs 示例代码")

请参考 workflow_handoffs 文档完成 Handoffs 示例代码


In [16]:
import tempfile
import subprocess
import sys
from agentscope.tool import ToolResponse
from agentscope.message import TextBlock

def execute_python_code_sync(
    code: str,
    timeout: float = 300,
) -> ToolResponse:
    """在本地执行 Python 代码的同步工具。"""
    with tempfile.TemporaryDirectory() as temp_dir:
        temp_file = os.path.join(temp_dir, "tmp.py")
        with open(temp_file, "w", encoding="utf-8") as f:
            f.write(code)
            
        env = os.environ.copy()
        env["PYTHONUTF8"] = "1"
        env["PYTHONIOENCODING"] = "utf-8"
        try:
            proc = subprocess.run(
                [sys.executable, "-u", temp_file],
                capture_output=True,
                text=True,
                timeout=timeout,
                env=env
            )
            returncode = proc.returncode
            stdout_str = proc.stdout
            stderr_str = proc.stderr
        except subprocess.TimeoutExpired as e:
            returncode = -1
            stdout_str = e.stdout.decode('utf-8') if e.stdout else ""
            stderr_str = e.stderr.decode('utf-8') if e.stderr else f"TimeoutError: The code execution exceeded the timeout of {timeout} seconds."

        return ToolResponse(
            content=[
                TextBlock(
                    type="text",
                    text=f"<returncode>{returncode}</returncode><stdout>{stdout_str}</stdout><stderr>{stderr_str}</stderr>",
                ),
            ],
        )

# 创建子智能体的工具函数
async def create_worker(
    task_description: str,
) -> ToolResponse:
    """创建一个子智能体来完成给定的任务。子智能体配备了 Python 执行工具。

    Args:
        task_description (``str``):
            子智能体要完成的任务描述。
    """
    # 为子智能体智能体配备一些工具
    toolkit = Toolkit()
    toolkit.register_tool_function(execute_python_code_sync)

    # 创建子智能体智能体
    worker = ReActAgent(
        name="Worker",
        sys_prompt="你是一个智能体。你的目标是完成给定的任务。",
        model=DashScopeChatModel(
            model_name="qwen-max",
            api_key=os.environ["DASHSCOPE_API_KEY"],
            stream=False,
        ),
        formatter=DashScopeChatFormatter(),
        toolkit=toolkit,
    )
    # 让子智能体完成任务
    res = await worker(Msg("user", task_description, "user"))
    return ToolResponse(
        content=res.get_content_blocks("text"),
    )

In [17]:
async def run_handoffs() -> None:
    """交接工作流示例。"""
    # 初始化协调者智能体
    toolkit = Toolkit()
    toolkit.register_tool_function(create_worker)

    orchestrator = ReActAgent(
        name="Orchestrator",
        sys_prompt="你是一个协调者智能体。你的目标是通过将任务分解为更小的任务并创建子智能体来完成它们，从而完成给定的任务。",
        model=DashScopeChatModel(
            model_name="qwen-max",
            api_key=os.environ["DASHSCOPE_API_KEY"],
            stream=False,
        ),
        memory=InMemoryMemory(),
        formatter=DashScopeChatFormatter(),
        toolkit=toolkit,
    )

    # 任务描述
    task_description = "在 Python 中执行 hello world"

    # 创建子智能体来完成任务
    await orchestrator(Msg("user", task_description, "user"))

In [18]:
await run_handoffs()

Orchestrator: {
    "type": "tool_use",
    "name": "create_worker",
    "input": {
        "task_description": "在 Python 中编写并执行一个打印 'Hello, World!' 的简单脚本。"
    },
    "id": "call_de69b5b815554dceb8cd24"
}
Worker: {
    "type": "tool_use",
    "name": "execute_python_code_sync",
    "input": {
        "code": "print('Hello, World!')"
    },
    "id": "call_2f464908b9fc41ea9f05ce"
}
system: {
    "type": "tool_result",
    "id": "call_2f464908b9fc41ea9f05ce",
    "name": "execute_python_code_sync",
    "output": [
        {
            "type": "text",
            "text": "<returncode>0</returncode><stdout>Hello, World!\n</stdout><stderr></stderr>"
        }
    ]
}
Worker: Python 脚本已成功执行，并打印出了 'Hello, World!'。
system: {
    "type": "tool_result",
    "id": "call_de69b5b815554dceb8cd24",
    "name": "create_worker",
    "output": [
        {
            "type": "text",
            "text": "Python 脚本已成功执行，并打印出了 'Hello, World!'。"
        }
    ]
}
Orchestrator: 子智能体已经成功地在 Python 中执行了 "Hell